In [1]:
experiment_definition = {
    "business_question": "Will SMS reminders increase appointment attendance?",
    "null_hypothesis": "Adding SMS reminders has no effect on appointment attendance rates",
    "alternative_hypothesis": "Adding SMS reminders increases appointment attendance rates",
    "minimum_detectable_effect": 0.05,  # 5% improvement needed to justify SMS costs
    "primary_metric": "appointment_attendance_rate",
    "secondary_metrics": [
        "patient_satisfaction_score",
        "provider_satisfaction_score",
        "support_contact_rate"
    ],
    "guardrail_metrics": [
        "app_crash_rate",
        "system_error_rate"
    ]
}

In [2]:
import hashlib
from datetime import datetime
from typing import Dict, List, Optional

class Experiment:
    def __init__(self, experiment_id: str, traffic_percentage: float = 0.5):
        self.experiment_id = experiment_id
        self.traffic_percentage = traffic_percentage
        self.start_time = datetime.now()
        
    def get_variant(self, user_id: str) -> str:
        """
        Deterministically assign users to variants using hashing
        
        Args:
            user_id: Unique identifier for the user
            
        Returns:
            str: 'control' or 'treatment'
        """
        # Create a deterministic hash combining user_id and experiment_id
        hash_input = f"{user_id}:{self.experiment_id}".encode('utf-8')
        hash_value = int(hashlib.sha256(hash_input).hexdigest(), 16)
        
        # Use the hash to determine variant
        is_in_experiment = (hash_value % 100) < (self.traffic_percentage * 100)
        if not is_in_experiment:
            return 'excluded'
            
        return 'treatment' if (hash_value % 2) == 0 else 'control'
        
    def is_eligible(self, user: Dict) -> bool:
        """
        Determine if a user is eligible for the experiment
        
        Args:
            user: Dictionary containing user information
            
        Returns:
            bool: Whether the user is eligible
        """
        return (
            user.get('has_phone_number', False) and  # Must have phone number
            user.get('communication_preferences', {}).get('sms_enabled', False) and  # SMS opted-in
            user.get('appointment_count', 0) > 0  # Not their first appointment
        )

class ReminderExperiment(Experiment):
    def __init__(self):
        super().__init__('reminder_optimization_2024Q1')
        self.metrics_calculator = MetricsCalculator()
        self.notification_system = NotificationSystem()
        
    def send_reminders(self, appointment: Dict) -> None:
        """
        Send reminders based on experiment variant
        
        Args:
            appointment: Dictionary containing appointment information
        """
        user_id = appointment['user_id']
        variant = self.get_variant(user_id)
        
        # Both groups get email
        self.notification_system.send_email(
            user_id=user_id,
            template='appointment_reminder',
            appointment_details=appointment
        )
        
        # Treatment group also gets SMS
        if variant == 'treatment':
            self.notification_system.send_sms(
                user_id=user_id,
                template='appointment_reminder_sms',
                appointment_details=appointment
            )
        
        # Log the reminder event
        self.log_reminder_event(user_id, variant, appointment['appointment_id'])

In [3]:
class MetricsCalculator:
    def __init__(self):
        self.db = DatabaseConnection()
        
    def calculate_attendance_rate(self, variant: str, 
                                start_date: datetime, 
                                end_date: datetime) -> float:
        """
        Calculate attendance rate for a variant
        
        Args:
            variant: 'control' or 'treatment'
            start_date: Start of analysis period
            end_date: End of analysis period
            
        Returns:
            float: Attendance rate
        """
        query = """
        SELECT 
            COUNT(CASE WHEN attended = TRUE THEN 1 END) * 100.0 / COUNT(*) as attendance_rate
        FROM appointments
        WHERE variant = %(variant)s
            AND appointment_time BETWEEN %(start_date)s AND %(end_date)s
        """
        
        return self.db.execute_query(query, {
            'variant': variant,
            'start_date': start_date,
            'end_date': end_date
        })
        
    def calculate_confidence_interval(self, 
                                   control_successes: int, 
                                   control_trials: int,
                                   treatment_successes: int, 
                                   treatment_trials: int,
                                   confidence_level: float = 0.95) -> Dict:
        """
        Calculate confidence interval for difference in proportions
        """
        from statsmodels.stats.proportion import proportion_effectsize
        from statsmodels.stats.proportion import proportion_confint
        
        # Calculate effect size
        effect_size = (treatment_successes/treatment_trials) - (control_successes/control_trials)
        
        # Calculate confidence interval
        ci_low, ci_high = proportion_confint(count=treatment_successes,
                                           nobs=treatment_trials,
                                           alpha=(1 - confidence_level))
        
        return {
            'effect_size': effect_size,
            'confidence_interval': (ci_low, ci_high)
        }

In [4]:
def analyze_experiment_results():
    calculator = MetricsCalculator()
    
    # Get results for both variants
    control_metrics = calculator.get_metrics('control')
    treatment_metrics = calculator.get_metrics('treatment')
    
    # Calculate primary metric impact
    attendance_impact = {
        'control_rate': control_metrics['attendance_rate'],
        'treatment_rate': treatment_metrics['attendance_rate'],
        'absolute_improvement': (
            treatment_metrics['attendance_rate'] - 
            control_metrics['attendance_rate']
        ),
        'relative_improvement': (
            (treatment_metrics['attendance_rate'] - 
             control_metrics['attendance_rate']) / 
            control_metrics['attendance_rate'] * 100
        )
    }
    
    # Calculate statistical significance
    stats_results = calculator.calculate_confidence_interval(
        control_metrics['attended'],
        control_metrics['total'],
        treatment_metrics['attended'],
        treatment_metrics['total']
    )
    
    # Calculate business impact
    business_impact = {
        'monthly_appointments': 10000,  # from business data
        'cost_per_missed_appointment': 45,  # dollars
        'projected_annual_savings': (
            10000 * 12 * attendance_impact['absolute_improvement'] * 45
        )
    }
    
    return {
        'metrics_impact': attendance_impact,
        'statistical_results': stats_results,
        'business_impact': business_impact
    }

In [7]:
def generate_experiment_report(results: Dict) -> str:
        """
        Generate a comprehensive experiment report
        """
        report = f"""
        A/B Test Results: Appointment Reminder Optimization
        ================================================

       Experiment Overview:
       - Duration: 4 weeks
       - Total Participants: {results['control_size'] + results['treatment_size']}
       - Control Group Size: {results['control_size']}
       - Treatment Group Size: {results['treatment_size']}

       Results:
       - Control Attendance Rate: {results['metrics_impact']['control_rate']:.1f}%
       - Treatment Attendance Rate: {results['metrics_impact']['treatment_rate']:.1f}%
       - Absolute Improvement: {results['metrics_impact']['absolute_improvement']:.1f}                    percentage points
       - Relative Improvement: {results['metrics_impact']['relative_improvement']:.1f}%

       Statistical Significance:
       - P-value: {results['statistical_results']['p_value']:.4f}
       - Confidence Interval: ({results['statistical_results']['confidence_interval'][0]:.1f}%, 
                           {results['statistical_results']['confidence_interval'][1]:.1f}%)

       Business Impact:
       - Projected Annual Savings: 
        ${results['business_impact']['projected_annual_savings']:,.2f}
       - Implementation Cost (SMS): ${results['business_impact']['sms_cost']:,.2f}
       - Net Annual Impact: ${results['business_impact']['net_impact']:,.2f}

       Recommendation:
       Based on the {results['metrics_impact']['absolute_improvement']:.1f} percentage point 
       improvement in attendance rates and projected annual savings of 
       ${results['business_impact']['net_impact']:,.2f}, we recommend implementing 
       SMS reminders for all appointments.

       Next Steps:
       1. Gradually roll out SMS reminders to all users over 4 weeks
       2. Monitor system performance and costs during rollout
       3. Conduct follow-up analysis after full rollout to verify results at scale
       """
    
        return report